# 17 · Citation Generation

**Idea:** grounded generation answers *only* from retrieved context and tags each sentence with its `[source]`. Citations make answers **verifiable** and failures **debuggable**: if a claim cites a source that was never retrieved, that is a hallucination signal. We generate cited answers, then validate the tags.

> Runs on Ollama. One cell optionally uses OpenAI — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

In [ ]:
from rag_lab import load_markdown, NumpyStore

store = NumpyStore(provider='ollama'); store.add(load_markdown())
query = 'What is reranking and why does it come after a first-stage retriever?'
hits = store.search(query, k=4)
print('retrieved sources:', sorted({h.chunk.source for h in hits}))

Generate an answer with inline citations, then check every cited tag against the sources we actually retrieved.

In [ ]:
from rag_lab import cited_answer
from study_buddy import CostTracker

tracker = CostTracker()
res = cited_answer(query, hits, provider='ollama', tracker=tracker)
print(res.answer)
print('\ncited:  ', res.cited)
print('valid:  ', res.valid)
print('invalid:', res.invalid, '(cited but not retrieved -> suspicious)')
print('cost:   ', tracker.summary())

Stress-test grounding: ask something the corpus **cannot** answer. A well-grounded model should decline rather than invent a citation.

In [ ]:
off_topic = 'What was the stock price of the company yesterday?'
res2 = cited_answer(off_topic, store.search(off_topic, k=3), provider='ollama')
print(res2.answer)
print('\ninvalid citations:', res2.invalid)

Optionally generate the cited answer with OpenAI. Skips if no key.

In [ ]:
try:
    cloud = CostTracker()
    r = cited_answer(query, hits, provider='openai', model='gpt-4o-mini', tracker=cloud)
    print(r.answer)
    print('\nvalid:', r.valid, '| invalid:', r.invalid)
    print('cost:', cloud.summary())
except Exception as e:
    print('openai skipped:', type(e).__name__, e)

**Takeaway:** citations turn a RAG answer into a *checkable* claim. Validating cited tags against retrieved sources is a cheap, automatic hallucination check — the seed of the evaluation harness in notebook 21 and the capstone.

That closes the Frontier RAG block (13–17). Next up in the roadmap → `18_document_ingestion.ipynb`.